# 🥑 Notebook 03: Approach 2 — Deep Learning (MobileNetV2 Transfer Learning)
## Avocado Ripening Stage Classification — Greenpods Labs Assignment 2

**Pipeline:**  
Image → Augmentation → MobileNetV2 (ImageNet backbone, frozen) → Custom Head → Cross-Entropy → Adam

**Why MobileNetV2?**
- Depthwise separable convolutions: high accuracy with low compute footprint
- Inverted residual structure preserves feature richness at bottleneck layers
- Pre-trained on 1.28M ImageNet images → rich low-level feature representations
- Suitable for edge deployment / automated sorting lines

---

## 1. Setup & Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import (
    load_image_paths_and_labels,
    stratified_split,
    get_transforms,
    AvocadoDataset,
    CLASS_NAMES,
)
from src.dl_model import (
    build_avocado_classifier,
    get_device,
    train_dl_model,
    evaluate_dl_model,
    plot_training_history,
    predict_single_image_dl,
    save_dl_model,
    load_dl_model,
)

plt.rcParams.update({'figure.dpi': 120})

DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'
MODELS_DIR   = PROJECT_ROOT / 'models'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = get_device()
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')


## 2. Data Loading & DataLoaders

In [ ]:
paths, labels, class_names = load_image_paths_and_labels(str(DATA_DIR))
print(f'Total images : {len(paths)}')
print(f'Classes      : {class_names}')

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(paths, labels)
print(f'\nSplit: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}')


In [ ]:
train_ds = AvocadoDataset(X_train, y_train, transform=get_transforms('train'))
val_ds   = AvocadoDataset(X_val,   y_val,   transform=get_transforms('val'))
test_ds  = AvocadoDataset(X_test,  y_test,  transform=get_transforms('test'))

BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'DataLoader batches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')


## 3. Visualise Augmented Training Samples

In [ ]:
import torchvision

imgs, lbls = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
imgs_denorm = (imgs * std + mean).clamp(0, 1)

grid = torchvision.utils.make_grid(imgs_denorm[:16], nrow=8, padding=2)
fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.set_title(f'Augmented Training Batch (labels: {[class_names[l] for l in lbls[:16].tolist()]})', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.show()


## 4. Model Architecture

In [ ]:
model = build_avocado_classifier(
    num_classes=len(class_names),
    freeze_backbone=True,
    dropout=0.3,
)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('MobileNetV2 Model Architecture:')
print(f'  Total parameters     : {total_params:,}')
print(f'  Trainable parameters : {trainable_params:,} ({trainable_params/total_params*100:.1f}%)')
print(f'  Frozen parameters    : {total_params - trainable_params:,}')
print(f'\nCustom Classifier Head:')
print(model.classifier)


## 5. Phase 1 Training — Frozen Backbone (Head Only)

In [ ]:
print('='*60)
print('PHASE 1: Training classifier head (backbone frozen)')
print('='*60)

model, history_phase1 = train_dl_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=20,
    lr=1e-3,
    weight_decay=1e-4,
    patience=5,
    device=DEVICE,
    save_path=str(MODELS_DIR / 'mobilenetv2_phase1.pth'),
)

plot_training_history(history_phase1, save_dir=str(PROCESSED_DIR))


## 6. Phase 2 Training — Fine-tune Full Model

In [ ]:
print('='*60)
print('PHASE 2: Fine-tuning full model (all layers)')
print('='*60)

for param in model.features.parameters():
    param.requires_grad = True

model, history_phase2 = train_dl_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=15,
    lr=1e-4, 
    weight_decay=1e-4,
    patience=5,
    device=DEVICE,
    save_path=str(MODELS_DIR / 'mobilenetv2_best.pth'),
)

plot_training_history(history_phase2, save_dir=str(PROCESSED_DIR))
print('\nBest model saved to models/mobilenetv2_best.pth')


## 7. Test Set Evaluation

In [ ]:
print('Loading best checkpoint and evaluating on TEST set...')

best_model = load_dl_model(
    str(MODELS_DIR / 'mobilenetv2_best.pth'),
    num_classes=len(class_names),
    device=DEVICE,
)

dl_test_results = evaluate_dl_model(
    model=best_model,
    test_loader=test_loader,
    class_names=class_names,
    device=DEVICE,
    model_name='MobileNetV2 (Test)',
    save_dir=str(PROCESSED_DIR),
)


## 8. Per-Class Performance Analysis

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = dl_test_results['y_true']
y_pred = dl_test_results['y_pred']

precisions = precision_score(y_true, y_pred, average=None, zero_division=0)
recalls    = recall_score(y_true, y_pred, average=None, zero_division=0)
f1s        = f1_score(y_true, y_pred, average=None, zero_division=0)

df_per_class = pd.DataFrame({
    'Class': class_names,
    'Precision': precisions,
    'Recall':    recalls,
    'F1-Score':  f1s,
})

import pandas as pd
print('Per-Class Metrics:')
print(df_per_class.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(class_names))
w = 0.25
ax.bar(x - w, precisions, w, label='Precision', color='steelblue')
ax.bar(x,     recalls,    w, label='Recall',    color='coral')
ax.bar(x + w, f1s,        w, label='F1-Score',  color='mediumseagreen')
ax.set_xticks(x); ax.set_xticklabels([c.capitalize() for c in class_names])
ax.set_title('MobileNetV2 — Per-Class Metrics', fontsize=13, fontweight='bold')
ax.set_ylabel('Score'); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'dl_per_class_metrics.png', dpi=150)
plt.show()


## 9. Inference Speed Benchmark

In [ ]:
import time

demo_path = X_test[0]
N_RUNS = 50

best_model.eval()
times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    predict_single_image_dl(best_model, demo_path, class_names, device=DEVICE)
    times.append((time.perf_counter() - t0) * 1000)

print(f'Inference Speed ({N_RUNS} runs):')
print(f'  Mean  : {np.mean(times):.1f} ms')
print(f'  Median: {np.median(times):.1f} ms')
print(f'  Min   : {np.min(times):.1f} ms')
print(f'  Max   : {np.max(times):.1f} ms')
print(f'  Device: {DEVICE}')


## 10. Approach 2 Summary

| Metric | Result |
|--------|--------|
| Architecture | MobileNetV2 (ImageNet pre-trained) |
| Input Size | 224×224 RGB |
| Training | 2-phase: head-only → full fine-tune |
| Loss | CrossEntropyLoss |
| Optimizer | Adam + ReduceLROnPlateau |
| Augmentation | Flip, Rotation±15°, ColorJitter, Translate |
| Test Accuracy | *see above* |
| Macro F1 | *see above* |
| Inference | *see above* ms |

**➡️ Next: `04_model_evaluation_and_demo.ipynb`**